# 02 — Silver : typage, nettoyage et référentiels

```
bronze/vols       ──typage + 8 règles──►  silver/vols        (une ligne par vol programmé, typée, partitionnée par mois)
                                     └──►  silver/rejets      (les lignes retirées, avec leur motif)
bronze/aeroports  ──filtre + jointure──►  silver/aeroports   (un aéroport par code IATA desservi : ville, état, lat/lon)
silver/vols       ──────────────────────►  silver/compagnies  (code → nom)
```

Ce que fait la couche silver :
1. **ne garder que les colonnes utiles** à la question du persona (la BTS en fournit 110) et les renommer en français ;
2. **typer** : dates, entiers, décimaux, booléens (le bronze est tout en texte) ;
3. **retirer les lignes incohérentes**, avec une règle nommée et justifiée pour chacune et le nombre de lignes retirées ;
4. **préparer les référentiels** (aéroports avec coordonnées, compagnies avec leur nom) pour le gold, la carte et l'ontologie.

Les vols **annulés** et **détournés** sont **conservés** : ce ne sont pas des erreurs, et le gold en a besoin
(taux d'annulation ; exclusion du dénominateur de ponctualité). Deux colonnes booléennes, `annule` et `detourne`,
permettent de les isoler.

## 1. Session Spark et lecture du bronze

In [1]:
import sys; sys.path.append("/home/jovyan/work")
from pyspark.sql import functions as F, Window
from commun.spark_utils import creer_session, format_tables, LAKE

spark = creer_session("02_silver")
assert format_tables(spark) == "delta", "Delta Lake n'est pas actif : Kernel → Restart."

bronze = spark.read.format("delta").load(f"{LAKE}/bronze/vols")
bronze_aeroports = spark.read.format("delta").load(f"{LAKE}/bronze/aeroports")
n_bronze = bronze.count()
print(f"bronze/vols : {n_bronze:,} lignes".replace(",", " "))

Préparation de Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3 … le premier démarrage peut prendre quelques minutes.


Session prête. Spark UI : http://localhost:4040


bronze/vols : 7 079 061 lignes


## 2. Sélection, renommage et typage

On garde 30 colonnes sur 110 (les colonnes `Div1…Div5` sur les aéroports de déroutement, les identifiants
internes BTS, etc. ne servent pas l'analyse). Les noms passent en `snake_case` français, comme dans le contrat gold.

Points de typage :
- les nombres BTS sont écrits `"-7.00"` : on les lit en `double`, puis en `int` quand ce sont des indicateurs (0/1) ;
- `CRSDepTime` (heure de départ **prévue**, format `HHMM` en heure locale ; `2400` = minuit) donne `heure_depart_prevue` (0–23),
  utilisée par la heatmap jour × heure ;
- `try_cast` renvoie `NULL` au lieu de planter si une valeur n'est pas convertible : les règles de la section 3 détectent ces cas.

In [2]:
d = lambda c: F.expr(f"try_cast({c} AS double)")
i = lambda c: F.expr(f"try_cast(try_cast({c} AS double) AS int)")

vols = bronze.select(
    F.expr("try_cast(FlightDate AS date)").alias("date_vol"),
    i("Year").alias("annee"),
    i("Month").alias("mois"),
    i("DayOfWeek").alias("jour_semaine"),                       # 1 = lundi … 7 = dimanche
    F.col("CRSDepTime").alias("_crs_dep_time"),                 # gardé pour la règle R3, retiré ensuite
    F.when(F.col("CRSDepTime") == "2400", 0)                   # convention BTS : 2400 = minuit
     .when(F.col("CRSDepTime").rlike(r"^([01]\d|2[0-3])[0-5]\d$"),
           F.substring("CRSDepTime", 1, 2).cast("int")).alias("heure_depart_prevue"),
    F.col("Reporting_Airline").alias("compagnie_code"),
    F.col("Flight_Number_Reporting_Airline").alias("numero_vol"),
    F.col("Tail_Number").alias("immatriculation"),
    F.col("Origin").alias("origine"),
    F.col("OriginCityName").alias("origine_ville"),
    F.col("OriginState").alias("origine_etat"),
    F.col("Dest").alias("destination"),
    F.col("DestCityName").alias("destination_ville"),
    F.col("DestState").alias("destination_etat"),
    d("Distance").alias("distance_miles"),
    d("DepDelay").alias("retard_depart_min"),
    d("ArrDelay").alias("retard_arrivee_min"),
    i("ArrDel15").alias("retard_15"),                           # 1 si arrivée avec 15 min de retard ou plus
    (d("Cancelled") == 1).alias("annule"),
    F.col("CancellationCode").alias("code_annulation"),         # A compagnie, B météo, C NAS, D sécurité
    (d("Diverted") == 1).alias("detourne"),
    d("AirTime").alias("temps_vol_min"),
    d("ActualElapsedTime").alias("duree_reelle_min"),
    d("CarrierDelay").alias("retard_compagnie_min"),
    d("WeatherDelay").alias("retard_meteo_min"),
    d("NASDelay").alias("retard_nas_min"),
    d("SecurityDelay").alias("retard_securite_min"),
    d("LateAircraftDelay").alias("retard_avion_min"),
)
vols.printSchema()

root
 |-- date_vol: date (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)
 |-- jour_semaine: integer (nullable = true)
 |-- _crs_dep_time: string (nullable = true)
 |-- heure_depart_prevue: integer (nullable = true)
 |-- compagnie_code: string (nullable = true)
 |-- numero_vol: string (nullable = true)
 |-- immatriculation: string (nullable = true)
 |-- origine: string (nullable = true)
 |-- origine_ville: string (nullable = true)
 |-- origine_etat: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- destination_ville: string (nullable = true)
 |-- destination_etat: string (nullable = true)
 |-- distance_miles: double (nullable = true)
 |-- retard_depart_min: double (nullable = true)
 |-- retard_arrivee_min: double (nullable = true)
 |-- retard_15: integer (nullable = true)
 |-- annule: boolean (nullable = true)
 |-- code_annulation: string (nullable = true)
 |-- detourne: boolean (nullable = true)
 |-- temps_vol_min: dou

## 3. Les règles de nettoyage

Chaque règle est une condition qui désigne les lignes **à retirer**. On calcule pour chaque ligne la **première règle**
qu'elle enfreint (colonne `_motif_rejet`) : une ligne n'est comptée qu'une fois, et tout se fait en une seule lecture
des 7 millions de lignes.

| Règle | Lignes visées | Pourquoi |
|---|---|---|
| R1 | date, compagnie, origine ou destination manquante ou illisible | sans ces clés, le vol ne peut être rattaché à aucune analyse |
| R2 | doublon sur la clé du vol (date, compagnie, n° de vol, origine, heure prévue) | un vol compté deux fois fausse tous les taux |
| R3 | heure de départ prévue invalide (hors `0000`–`2359` ; `2400` est accepté : c'est la notation BTS de minuit) | nécessaire à la heatmap jour × heure |
| R4 | vol opéré sans retard à l'arrivée connu | on ne peut dire ni « à l'heure » ni « en retard » |
| R5 | retard à l'arrivée supérieur à 24 h | valeurs extrêmes (jusqu'à plus de 60 h) : quelques centaines de vols qui tirent les moyennes vers le haut, alors que le persona pilote le retard courant |
| R6 | avance à l'arrivée de plus de 90 min | physiquement improbable sur un vol intérieur : erreur de saisie probable |
| R7 | indicateur `retard_15` incohérent avec le retard en minutes | la ponctualité repose sur cet indicateur : il doit valoir 1 exactement quand le retard est ≥ 15 min |
| R8 | vol en retard (≥ 15 min) dont la somme des 5 causes ≠ retard à l'arrivée | la BTS impose que les causes répartissent tout le retard ; sinon la part par cause (gold) serait fausse |

« Opéré » = ni annulé, ni détourné (ces vols n'ont pas d'heure d'arrivée à destination, c'est normal).

In [3]:
causes = ["retard_compagnie_min", "retard_meteo_min", "retard_nas_min", "retard_securite_min", "retard_avion_min"]
somme_causes = sum(F.coalesce(F.col(c), F.lit(0.0)) for c in causes)
opere = ~F.col("annule") & ~F.col("detourne")
retard = F.col("retard_arrivee_min")

cle_vol = ["date_vol", "compagnie_code", "numero_vol", "origine", "_crs_dep_time"]
rang_doublon = F.row_number().over(Window.partitionBy(*cle_vol).orderBy(F.lit(1)))

REGLES = [   # (code, libellé, condition de rejet) — appliquées dans cet ordre
    ("R1", "clé du vol manquante ou illisible",
        F.col("date_vol").isNull() | F.col("compagnie_code").isNull()
        | F.col("origine").isNull() | F.col("destination").isNull()),
    ("R2", "doublon sur la clé du vol", rang_doublon > 1),
    ("R3", "heure de départ prévue invalide", F.col("heure_depart_prevue").isNull()),
    ("R4", "vol opéré sans retard à l'arrivée", opere & retard.isNull()),
    ("R5", "retard à l'arrivée > 24 h", retard > 24 * 60),
    ("R6", "avance à l'arrivée > 90 min", retard < -90),
    ("R7", "retard_15 incohérent avec le retard",
        opere & (((F.col("retard_15") == 1) & (retard < 15)) | ((F.col("retard_15") == 0) & (retard >= 15)))),
    ("R8", "somme des causes ≠ retard à l'arrivée",
        opere & (retard >= 15) & (F.abs(somme_causes - retard) > 0.5)),
]

motif = F.lit(None).cast("string")
for code_regle, _, condition in reversed(REGLES):     # la première règle enfreinte l'emporte
    motif = F.when(condition, F.lit(code_regle)).otherwise(motif)

vols_motif = vols.withColumn("_motif_rejet", motif)

## 4. Écriture de silver/vols et silver/rejets

On écrit d'abord toutes les lignes avec leur motif dans une table temporaire, pour ne calculer les règles
(dont la fenêtre des doublons) qu'une seule fois ; silver et rejets en sont ensuite deux filtres.

Avant d'écrire silver partitionné par mois, on regroupe les lignes par mois (`repartition("mois")`) : sans cela,
chaque tâche Spark garde un fichier ouvert par mois rencontré (jusqu'à 12), ce qui fait monter la mémoire près
de la limite des 8 Go de Docker.

In [4]:
TMP = f"{LAKE}/tmp/vols_motif"
vols_motif.write.format("delta").mode("overwrite").save(TMP)
tout = spark.read.format("delta").load(TMP)

(tout.filter(F.col("_motif_rejet").isNull())
     .drop("_motif_rejet", "_crs_dep_time")
     .repartition("mois")      # une tâche par mois : un seul fichier ouvert à la fois, mémoire maîtrisée
     .write.format("delta").mode("overwrite").option("overwriteSchema", True)
     .partitionBy("mois").save(f"{LAKE}/silver/vols"))

(tout.filter(F.col("_motif_rejet").isNotNull())
     .write.format("delta").mode("overwrite").option("overwriteSchema", True)
     .save(f"{LAKE}/silver/rejets"))
print("silver/vols et silver/rejets écrits.")

silver/vols et silver/rejets écrits.


### Tableau des règles : lignes retirées

C'est le tableau demandé par l'énoncé (repris dans le README). Une règle à **0** n'est pas inutile : elle prouve
que le cas a été contrôlé. Les données BTS sont publiées après validation par le DOT, d'où peu d'erreurs.

In [5]:
import pandas as pd

comptes = {r["_motif_rejet"]: r["count"] for r in tout.groupBy("_motif_rejet").count().collect()}
tableau = pd.DataFrame(
    [(c, lib, comptes.get(c, 0)) for c, lib, _ in REGLES],
    columns=["règle", "description", "lignes retirées"])
tableau["% du bronze"] = (100 * tableau["lignes retirées"] / n_bronze).round(4)

silver = spark.read.format("delta").load(f"{LAKE}/silver/vols")
n_silver = silver.count()
assert n_silver + tableau["lignes retirées"].sum() == n_bronze, "Des lignes ont disparu sans motif !"

print(f"bronze : {n_bronze:,} → silver : {n_silver:,} lignes "
      f"({n_bronze - n_silver:,} retirées, {100 * (n_bronze - n_silver) / n_bronze:.3f} %)".replace(",", " "))
tableau

bronze : 7 079 061 → silver : 7 078 487 lignes (574 retirées  0.008 %)


,règle,description,lignes retirées,% du bronze
0,R1,clé du vol manquante ou illisible,0,0.0000
1,R2,doublon sur la clé du vol,0,0.0000
2,R3,heure de départ prévue invalide,0,0.0000
3,R4,vol opéré sans retard à l'arrivée,0,0.0000
4,R5,retard à l'arrivée > 24 h,567,0.0080
5,R6,avance à l'arrivée > 90 min,5,0.0001
6,R7,retard_15 incohérent avec le retard,0,0.0000
7,R8,somme des causes ≠ retard à l'arrivée,2,0.0000


In [6]:
# Exemples de lignes rejetées, une par règle déclenchée
rejets = spark.read.format("delta").load(f"{LAKE}/silver/rejets")
(rejets.withColumn("n", F.row_number().over(Window.partitionBy("_motif_rejet").orderBy("date_vol")))
       .filter("n = 1")
       .select("_motif_rejet", "date_vol", "compagnie_code", "origine", "destination", "_crs_dep_time",
               "retard_arrivee_min", "retard_15", *causes)
       .orderBy("_motif_rejet").show(truncate=False))

+------------+----------+--------------+-------+-----------+-------------+------------------+---------+--------------------+----------------+--------------+-------------------+----------------+
|_motif_rejet|date_vol  |compagnie_code|origine|destination|_crs_dep_time|retard_arrivee_min|retard_15|retard_compagnie_min|retard_meteo_min|retard_nas_min|retard_securite_min|retard_avion_min|
+------------+----------+--------------+-------+-----------+-------------+------------------+---------+--------------------+----------------+--------------+-------------------+----------------+
|R5          |2024-01-04|AA            |STX    |MIA        |1531         |1676.0            |1        |1647.0              |0.0             |1.0           |0.0                |28.0            |
|R6          |2024-02-02|DL            |JFK    |SEA        |1900         |-91.0             |0        |NULL                |NULL            |NULL          |NULL               |NULL            |
|R8          |2024-07-10|F9   

In [7]:
# La table temporaire n'est plus utile : on supprime ses fichiers du data lake
chemin_tmp = spark._jvm.org.apache.hadoop.fs.Path(TMP)
chemin_tmp.getFileSystem(spark._jsc.hadoopConfiguration()).delete(chemin_tmp, True)
print("tmp/vols_motif supprimée.")

tmp/vols_motif supprimée.


### Contrôles sur silver

Le contrat gold suppose : des mois 1 à 12, des jours 1 à 7, des heures 0 à 23, et que les vols annulés et détournés
soient bien présents.

In [8]:
silver.select(
    F.min("date_vol").alias("premier_vol"), F.max("date_vol").alias("dernier_vol"),
    F.countDistinct("mois").alias("mois"), F.countDistinct("jour_semaine").alias("jours"),
    F.min("heure_depart_prevue").alias("heure_min"), F.max("heure_depart_prevue").alias("heure_max"),
    F.sum(F.col("annule").cast("int")).alias("annules"),
    F.sum(F.col("detourne").cast("int")).alias("detournes"),
    F.round(100 * F.avg(F.when(~F.col("annule") & ~F.col("detourne"), 1 - F.col("retard_15"))), 1).alias("pct_a_l_heure"),
).show()

+-----------+-----------+----+-----+---------+---------+-------+---------+-------------+
|premier_vol|dernier_vol|mois|jours|heure_min|heure_max|annules|detournes|pct_a_l_heure|
+-----------+-----------+----+-----+---------+---------+-------+---------+-------------+
| 2024-01-01| 2024-12-31|  12|    7|        0|       23|  96315|    17499|         79.2|
+-----------+-----------+----+-----+---------+---------+-------+---------+-------------+



## 5. Référentiel des aéroports → `silver/aeroports`

Pour chaque code IATA présent dans les vols (origine ou destination), on récupère :
- **ville et état** depuis les vols BTS (`origine_ville` est de la forme `"Dallas/Fort Worth, TX"` : on garde la partie avant la virgule) ;
- **nom, latitude, longitude, type, page Wikipédia** depuis OurAirports (jointure sur `iata_code`).

**Correction de référentiel** : l'aéroport de West Palm Beach (`PBI` dans les données BTS 2024) a été renommé en
2025 et OurAirports le liste désormais sous le code IATA `DJT`. Sans correction, ses 29 476 départs n'auraient pas
de coordonnées. On garde le code BTS `PBI` et on lit les coordonnées sous l'ancien code dans une petite table de
correspondance, plutôt que de retirer ces vols.

In [9]:
def en_map(dictionnaire):
    """Dictionnaire Python → colonne map Spark, utilisable comme table de correspondance.

    On évite spark.createDataFrame(liste) pour ces petites tables : en Spark 4 il démarre un worker Python
    par cœur (environ 360 Mo chacun), qui restent en mémoire jusqu'à la fin de la session.
    """
    return F.create_map([F.lit(x) for paire in dictionnaire.items() for x in paire])


# code BTS (2024) → code IATA actuel dans OurAirports
CORRESPONDANCE_IATA = {"PBI": "DJT"}

dessertes = (silver.select(F.col("origine").alias("code_iata"), F.col("origine_ville").alias("ville_bts"),
                           F.col("origine_etat").alias("etat"))
             .unionByName(silver.select(F.col("destination").alias("code_iata"),
                                        F.col("destination_ville").alias("ville_bts"),
                                        F.col("destination_etat").alias("etat")))
             .groupBy("code_iata").agg(F.first("ville_bts").alias("ville_bts"), F.first("etat").alias("etat")))

ref = (bronze_aeroports
       .filter(F.col("iata_code").isNotNull())
       .select(F.col("iata_code").alias("code_ourairports"),
               F.col("name").alias("nom"),
               F.col("type").alias("type_ourairports"),
               F.expr("try_cast(latitude_deg AS double)").alias("lat"),
               F.expr("try_cast(longitude_deg AS double)").alias("lon"),
               F.col("wikipedia_link").alias("wikipedia"))
       .dropDuplicates(["code_ourairports"]))

aeroports = (dessertes
             .withColumn("code_ourairports",
                         F.coalesce(en_map(CORRESPONDANCE_IATA)[F.col("code_iata")], F.col("code_iata")))
             .join(ref, "code_ourairports", "left")
             .select("code_iata",
                     F.trim(F.split("ville_bts", ",").getItem(0)).alias("ville"),
                     "etat", "nom", "type_ourairports", "lat", "lon", "wikipedia")
             .orderBy("code_iata"))

aeroports.write.format("delta").mode("overwrite").option("overwriteSchema", True).save(f"{LAKE}/silver/aeroports")
aeroports = spark.read.format("delta").load(f"{LAKE}/silver/aeroports")

sans_coord = aeroports.filter(F.col("lat").isNull())
print(f"silver/aeroports : {aeroports.count()} aéroports desservis, dont {sans_coord.count()} sans coordonnées")
assert sans_coord.count() == 0, sans_coord.collect()
aeroports.filter(F.col("code_iata").isin("ATL", "DFW", "PBI", "JFK", "HNL")).show(truncate=40)

silver/aeroports : 348 aéroports desservis, dont 0 sans coordonnées


+---------+--------------------------+----+----------------------------------------+----------------+---------+----------+----------------------------------------+
|code_iata|                     ville|etat|                                     nom|type_ourairports|      lat|       lon|                               wikipedia|
+---------+--------------------------+----+----------------------------------------+----------------+---------+----------+----------------------------------------+
|      ATL|                   Atlanta|  GA|Hartsfield Jackson Atlanta Internatio...|   large_airport|  33.6367|-84.428101|https://en.wikipedia.org/wiki/Hartsfi...|
|      DFW|         Dallas/Fort Worth|  TX| Dallas Fort Worth International Airport|   large_airport|32.896801|-97.038002|https://en.wikipedia.org/wiki/Dallas/...|
|      HNL|                  Honolulu|  HI|  Daniel K. Inouye International Airport|   large_airport|21.318387|-157.92567|https://en.wikipedia.org/wiki/Daniel_...|
|      JFK|     

## 6. Référentiel des compagnies → `silver/compagnies`

Les vols ne donnent que le code de la compagnie (`Reporting_Airline`). Les noms viennent de la table de référence
BTS `L_UNIQUE_CARRIERS` (https://www.transtats.bts.gov/), recopiée ici pour les 15 compagnies présentes en 2024.
On vérifie qu'aucun code des vols n'est oublié.

In [10]:
NOMS_COMPAGNIES = {
    "AA": "American Airlines", "AS": "Alaska Airlines", "B6": "JetBlue Airways", "DL": "Delta Air Lines",
    "F9": "Frontier Airlines", "G4": "Allegiant Air", "HA": "Hawaiian Airlines", "MQ": "Envoy Air",
    "NK": "Spirit Airlines", "OH": "PSA Airlines", "OO": "SkyWest Airlines", "UA": "United Air Lines",
    "WN": "Southwest Airlines", "YX": "Republic Airline", "9E": "Endeavor Air",
}

compagnies = (silver.groupBy("compagnie_code").agg(F.count("*").alias("vols"))
              .withColumn("compagnie_nom", en_map(NOMS_COMPAGNIES)[F.col("compagnie_code")])
              .orderBy(F.desc("vols")))

oublies = [r["compagnie_code"] for r in compagnies.filter(F.col("compagnie_nom").isNull()).collect()]
assert not oublies, f"Compagnies sans nom : {oublies}"

compagnies.select("compagnie_code", "compagnie_nom").write.format("delta").mode("overwrite") \
          .option("overwriteSchema", True).save(f"{LAKE}/silver/compagnies")
compagnies.show()

+--------------+-------+------------------+
|compagnie_code|   vols|     compagnie_nom|
+--------------+-------+------------------+
|            WN|1419419|Southwest Airlines|
|            DL|1009173|   Delta Air Lines|
|            AA| 983880| American Airlines|
|            UA| 760440|  United Air Lines|
|            OO| 744634|  SkyWest Airlines|
|            YX| 301465|  Republic Airline|
|            MQ| 279940|         Envoy Air|
|            NK| 261101|   Spirit Airlines|
|            AS| 245819|   Alaska Airlines|
|            B6| 240264|   JetBlue Airways|
|            OH| 227904|      PSA Airlines|
|            F9| 208622| Frontier Airlines|
|            9E| 200094|      Endeavor Air|
|            G4| 117204|     Allegiant Air|
|            HA|  78528| Hawaiian Airlines|
+--------------+-------+------------------+



## C'est réussi quand…

- la console RustFS montre `silver/vols` (un dossier par mois), `silver/rejets`, `silver/aeroports`, `silver/compagnies` ;
- le tableau des règles indique, pour chacune, le nombre de lignes retirées, et bronze = silver + rejets ;
- tous les aéroports desservis ont des coordonnées et toutes les compagnies un nom.

Suite : `03_gold` agrège `silver/vols` en tables prêtes pour le dashboard et les exporte dans PostgreSQL.

In [11]:
spark.stop()